# Ocean Surface Mixed Layer Depths

Mixed layer depth comparison with Argo observations using the energy-based
(PE anomaly) MLD definition.

Authors: Brandon Reichl and John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variables: MLD from PE anomaly method
# MLD_EN1 (25 J/m\u00b2 threshold) captures shallow/minimum mixed layers
# MLD_EN2 (2500 J/m\u00b2 threshold) captures deep/maximum mixed layers
variables = [
    RequestedVariable("MLD_EN1", "ocean_month"),
    RequestedVariable("MLD_EN2", "ocean_month"),
]

# Diagnostic configuration
diag_name = "Mixed Layer Depths"
diag_desc = "Ocean mixed layer depth comparison with Argo observations"
user_options = {}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-516", date_range=("1993-01-01", "2017-12-31"), name="OM5 B11 NB"),
    CaseGroup2("odiv-290", date_range=("1993-01-01", "2017-12-31"), name="OM5 B01 (OM4-like)"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

## Main Diagnostic

In [ ]:
import os

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xesmf as xe

os.environ["MOMGRID_WEIGHTS_DIR"] = "/nbhome/jpk/grid_weights"

In [ ]:
convert_to_momgrid(diag, return_corners=True)

In [ ]:
all_figs = []
climplot.publication()

In [ ]:
# Load Argo MLD observations (PE anomaly method)
obs_en1_path = "/archive/ogrp/MAR/data/MLDs/Argo_MLD.mld_pe_anomaly_25.nc"
obs_en2_path = "/archive/ogrp/MAR/data/MLDs/Argo_MLD.mld_pe_anomaly_2500.nc"

ds_en1_obs = xr.open_dataset(obs_en1_path)
ds_en2_obs = xr.open_dataset(obs_en2_path)

ds_obsdata = xr.Dataset()
ds_obsdata["lat"] = ds_en1_obs["Lat"].values
ds_obsdata["lon"] = ds_en1_obs["Lon"].values
ds_obsdata["Month"] = ds_en1_obs["Month"].values
ds_obsdata["MLD_EN1"] = (("Month", "lon", "lat"), ds_en1_obs["MLD_mean"].values)
ds_obsdata["MLD_EN2"] = (("Month", "lon", "lat"), ds_en2_obs["MLD_mean"].values)

# Obs min/max MLD across monthly climatology
obs_mld_min = ds_obsdata.MLD_EN1.min(dim="Month", skipna=False).T
obs_mld_max = ds_obsdata.MLD_EN2.max(dim="Month", skipna=False).T

In [ ]:
# Domain bounds for trimming obs grid
Dims = [-240, 120, -90, 90]

# Prepare observation grid (sort and trim longitudes)
obs_lat = np.copy(ds_obsdata["lat"].values)
obs_lon = np.copy(ds_obsdata["lon"].values)
obs_lon[obs_lon < Dims[0]] += 360
obs_lon[obs_lon > Dims[1]] -= 360

xi = np.argsort(obs_lon)
obs_lon_sort = obs_lon[xi]
lonlims = np.where((obs_lon_sort > Dims[0]) & (obs_lon_sort < Dims[1]))[0]
latlims = np.where((obs_lat > Dims[2]) & (obs_lat < Dims[3]))[0]
cmn_lat = obs_lat[latlims]
cmn_lon = obs_lon_sort[lonlims]

# Trimmed obs arrays on sorted grid
obs_min_arr = (obs_mld_min.values[:, xi])[
    latlims[0] : latlims[-1] + 1, lonlims[0] : lonlims[-1] + 1
]
obs_max_arr = (obs_mld_max.values[:, xi])[
    latlims[0] : latlims[-1] + 1, lonlims[0] : lonlims[-1] + 1
]


def compute_mld_stats(model_vals, obs_vals, lat_vals, lon_vals):
    """Compute area-weighted bias, RMS, and r-squared."""
    plon, plat = np.meshgrid(lon_vals, lat_vals)
    mask = np.isfinite(model_vals) & np.isfinite(obs_vals)
    diff = model_vals - obs_vals
    cos_wt = np.cos(plat * np.pi / 180.0)
    bias = np.nansum(diff[mask] * cos_wt[mask]) / np.nansum(cos_wt[mask])
    rms = np.sqrt(
        np.nansum(diff[mask] ** 2 * cos_wt[mask]) / np.nansum(cos_wt[mask])
    )
    r2 = np.corrcoef(model_vals[mask].ravel(), obs_vals[mask].ravel())[1, 0] ** 2
    return {
        "bias": round(float(bias), 4),
        "rms": round(float(rms), 4),
        "r2": round(float(r2), 4),
    }


# Process each experiment
results = {}
native_mld = {}
ds_target = xr.Dataset({"lat": cmn_lat, "lon": cmn_lon})

for group in diag.groups:
    var_en1 = diag.varmap["MLD_EN1"]
    var_en2 = diag.varmap["MLD_EN2"]
    ds_en1 = group.datasets[var_en1]
    ds_en2 = group.datasets[var_en2]

    # Monthly climatology
    clim_en1 = ds_en1.MLD_EN1.groupby("time.month").mean("time")
    clim_en2 = ds_en2.MLD_EN2.groupby("time.month").mean("time")

    # Min MLD from EN1, Max MLD from EN2
    mld_min = clim_en1.min(dim="month")
    mld_max = clim_en2.max(dim="month")

    # Store native grid data for absolute value plots
    native_mld[group] = {"min": mld_min, "max": mld_max}

    # Build dataset for regridding
    model_ds = xr.Dataset()
    model_ds["yh"] = ds_en1.yh.values
    model_ds["xh"] = ds_en1.xh.values
    model_ds["MLD_min"] = (("yh", "xh"), mld_min.values)
    model_ds["MLD_max"] = (("yh", "xh"), mld_max.values)
    model_ds = model_ds.assign_coords(
        lon=(("yh", "xh"), ds_en1.geolon.values),
        lat=(("yh", "xh"), ds_en1.geolat.values),
    )

    # Regrid model to observation grid
    regridder = xe.Regridder(model_ds, ds_target, "bilinear", periodic=True)
    mod_min = regridder(model_ds.MLD_min.data)
    mod_max = regridder(model_ds.MLD_max.data)

    # Compute statistics
    stats_min = compute_mld_stats(mod_min, obs_min_arr, cmn_lat, cmn_lon)
    stats_max = compute_mld_stats(mod_max, obs_max_arr, cmn_lat, cmn_lon)

    results[group] = {
        "min": {"model": mod_min, "obs": obs_min_arr, "stats": stats_min},
        "max": {"model": mod_max, "obs": obs_max_arr, "stats": stats_max},
    }

    # Register metrics
    for method in ["min", "max"]:
        for k, v in results[group][method]["stats"].items():
            group.add_metric(f"mld_{method}", (k, float(v)))

In [ ]:
lon_grid, lat_grid = np.meshgrid(cmn_lon, cmn_lat)
projection = ccrs.Robinson(central_longitude=-60)

for method, label in [("min", "Minimum"), ("max", "Maximum")]:
    clim = [10, 60] if method == "min" else [10, 500]
    step_abs = 5 if method == "min" else 50
    diff_range = 20 if method == "min" else 100
    delta_diff = 2 if method == "min" else 10

    nexps = len(diag.groups)

    # --- Figure: Model absolute values on native grid ---
    fig, axes = climplot.panel_figure(1, nexps, projection=projection)

    cmap_abs, norm_abs, levels_abs = climplot.sequential_cmap(
        clim[0], clim[1], step_abs, cmap_name="viridis_r",
    )

    for n, group in enumerate(diag.groups):
        ax = axes.flat[n]
        ax.set_global()
        varname = "MLD_EN1" if method == "min" else "MLD_EN2"
        var = diag.varmap[varname]
        ds = group.datasets[var]
        cb = climplot.plot_ocean_field(
            ax, ds.geolon_c.values, ds.geolat_c.values,
            native_mld[group][method].values,
            cmap=cmap_abs, norm=norm_abs,
        )
        ax.set_title(group.name)

    climplot.add_panel_labels(axes.flatten())
    climplot.bottom_colorbar(cb, fig, axes, f"Monthly {label} MLD [m]",
                             extend="both", fraction=0.06)
    all_figs.append(fig)

    # --- Figure: Bias (Model - Argo) on regridded grid ---
    fig, axes = climplot.panel_figure(1, nexps, projection=projection)

    cmap_diff, norm_diff, levels_diff = climplot.discrete_cmap(
        -diff_range, diff_range, delta_diff, cmap_name="PuOr",
        center_on_white=True,
    )

    for n, group in enumerate(diag.groups):
        ax = axes.flat[n]
        climplot.add_land_feature(ax)
        ax.set_global()
        data = results[group][method]
        diff = data["model"] - data["obs"]
        stats = data["stats"]
        cb = ax.pcolormesh(
            lon_grid, lat_grid, diff,
            shading="auto",
            transform=ccrs.PlateCarree(),
            cmap=cmap_diff, norm=norm_diff,
        )
        ax.set_title(group.name)

        stats_str = (
            f"bias = {stats['bias']:.3f} m   "
            f"r\u00b2 = {stats['r2']:.3f}"
        )
        ax.text(
            0.5, -0.1, stats_str, ha="center", style="italic",
            transform=ax.transAxes, fontsize=5,
        )

    climplot.add_panel_labels(axes.flatten())
    climplot.bottom_colorbar(cb, fig, axes,
                             f"Monthly {label} MLD Bias (Model - Argo) [m]",
                             extend="both", fraction=0.06)
    all_figs.append(fig)

### Write Metrics to File

In [ ]:
diag.write_metrics("MLD_metrics.json")

### Make a PowerPoint Presentation of Figures

In [ ]:
nbtools.save_pptx(all_figs, "MLD_mixed_layer_depths.pptx")